In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

In [ ]:
os.getcwd()

In [2]:
df_og=pd.read_csv('c:\\Users\\Ducks\\Downloads\\ducks python\\fraud-detection-system\\data\\creditcard.csv')

In [3]:
df_og.head()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


In [ ]:
df=df_og.copy()
df.shape

In [ ]:
df.isnull().sum()

In [ ]:
df.dtypes

In [ ]:
class_0=df['Class'].value_counts()[0]
df_len=len(df)
class_1=df['Class'].value_counts()[1]
print((class_0*100)/df_len)
print((class_1*100)/df_len)

In [ ]:
plt.bar(['legitimate','fraud'],[class_0,class_1],color='blue')
plt.xlabel('type of transaction')
plt.ylabel('number of transactions')
plt.yscale('log')
plt.title('fraud vs legitimate transaction')

In [ ]:
lst0=[]
lst1=[]
for i in range(len(df)):
    if df['Class'][i]==0:
        lst0.append(df['Amount'][i])
    else:
        lst1.append(df['Amount'][i])

In [ ]:
print(np.mean(lst0))
print(np.median(lst0))
print(np.min(lst0))
print(np.max(lst0))

In [ ]:
print(np.mean(lst1))
print(np.median(lst1))
print(np.min(lst1))
print(np.max(lst1))

In [ ]:
legitimate_amnt=df[df['Class']==0]['Amount']
fraud_amnt=df[df['Class']==1]['Amount']

In [ ]:
plt.hist(legitimate_amnt,bins=50)

In [ ]:
plt.hist(fraud_amnt,bins=50)


In [ ]:
legit_time = df[df['Class'] == 0]['Time']
fraud_time = df[df['Class'] == 1]['Time']

In [ ]:
plt.hist(legit_time,bins=60,edgecolor='black')

In [ ]:
plt.hist(fraud_time,bins=60,edgecolor='black')

In [ ]:
#find highly correlated features wtr class
corr=df.corr()
corr['Class'].sort_values()

In [ ]:
class_corr = corr['Class'].drop('Class')
plt.figure(figsize=(12,6))

plt.bar(class_corr.index, class_corr.values)

plt.xticks(rotation=90)
plt.xlabel('Features')
plt.ylabel('Correlation with Class')
plt.title('Feature Correlation with Fraud')

plt.show()

In [ ]:
y=df['Class']
X=df.drop('Class',axis=1)

In [ ]:
print(X.shape)
print(y.shape)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
print(X_train.shape)
print(X_test.shape)

print(y_train.value_counts(normalize=True))
print(y_test.value_counts(normalize=True))

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
from sklearn.linear_model import LogisticRegression
model_lr=LogisticRegression(max_iter=1000)
model_lr.fit(X_train_scaled,y_train)
y_pred_lr=model_lr.predict(X_test)

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred_lr)

print(cm)

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred_lr))

In [ ]:
model_lr_balanced = LogisticRegression(
    max_iter=1000,
    class_weight='balanced'
)

model_lr_balanced.fit(X_train_scaled, y_train)

y_pred_lr_balanced = model_lr_balanced.predict(X_test_scaled)

In [ ]:
print(confusion_matrix(y_test, y_pred_lr_balanced))

print(classification_report(y_test, y_pred_lr_balanced))

In [ ]:
from imblearn.over_sampling import SMOTE
smote=SMOTE(random_state=42)
X_train_smote,y_train_smote=smote.fit_resample(X_train_scaled,y_train)

In [ ]:
print(y_train.value_counts())
print(y_train_smote.value_counts())

In [ ]:
model_lr_smote=LogisticRegression(max_iter=1000)
model_lr_smote.fit(X_train_smote,y_train_smote)
y_pred_lr_smote=model_lr_smote.predict(X_test_scaled)

In [ ]:
print(confusion_matrix(y_test,y_pred_lr_smote))
print(classification_report(y_test,y_pred_lr_smote))

In [ ]:
from sklearn.ensemble import RandomForestClassifier

model_rf = RandomForestClassifier(
    n_estimators=200,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

model_rf.fit(X_train, y_train)

y_pred_rf = model_rf.predict(X_test)

In [ ]:
print(confusion_matrix(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))

In [ ]:
from xgboost import XGBClassifier

model_xgb = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    random_state=42,
    eval_metric='logloss'
)

model_xgb.fit(X_train, y_train)

y_pred_xgb = model_xgb.predict(X_test)

In [ ]:
print(confusion_matrix(y_test, y_pred_xgb))
print(classification_report(y_test, y_pred_xgb))


In [ ]:
y_prob_xgb=model_xgb.predict_proba(X_test)[:,1]

In [ ]:
from sklearn.metrics import average_precision_score
pr_auc_xgb=average_precision_score(y_test,y_prob_xgb)
print(pr_auc_xgb)

In [ ]:
threshold = 0.4
y_pred_04 = (y_prob_xgb >= threshold).astype(int)

print(confusion_matrix(y_test, y_pred_04))
print(classification_report(y_test, y_pred_04))

In [ ]:
threshold = 0.3
y_pred_03 = (y_prob_xgb >= threshold).astype(int)

print(confusion_matrix(y_test, y_pred_03))
print(classification_report(y_test, y_pred_03))

In [ ]:
threshold = 0.2
y_pred_02 = (y_prob_xgb >= threshold).astype(int)

print(confusion_matrix(y_test, y_pred_02))
print(classification_report(y_test, y_pred_02))

In [ ]:
X_train_final, X_val, y_train_final, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.2,
    random_state=42,
    stratify=y_train
)

In [ ]:
model_xgb_final = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    random_state=42,
    eval_metric='logloss'
)

model_xgb_final.fit(X_train_final, y_train_final)

In [ ]:
y_prob_val = model_xgb_final.predict_proba(X_val)[:, 1]

In [ ]:
thresholds = [0.5, 0.3, 0.2]

for t in thresholds:
    y_pred_val = (y_prob_val >= t).astype(int)

    print("Threshold:", t)
    print(confusion_matrix(y_val, y_pred_val))
    print(classification_report(y_val, y_pred_val))
    print("-" * 70)

In [ ]:
y_prob_test_final = model_xgb_final.predict_proba(X_test)[:, 1]

y_pred_test_final = (y_prob_test_final >= 0.2).astype(int)

print(confusion_matrix(y_test, y_pred_test_final))
print(classification_report(y_test, y_pred_test_final))

In [ ]:
print(type(X_train_final))
print(type(X_val))

print(X_train_final.shape)
print(X_val.shape)

In [ ]:
import shap 
explainer = shap.TreeExplainer(model_xgb_final)
X_val_shap = X_val.sample(n=5000, random_state=42)
shap_values=explainer(X_val_shap)

In [ ]:
shap.plots.bar(shap_values)

In [ ]:
shap.plots.beeswarm(shap_values)

In [ ]:
single_row = X_val_shap.iloc[[0]]
single_shap = explainer(single_row)
print(single_shap.shape)

In [ ]:
shap.plots.waterfall(single_shap[0])

In [ ]:
fraud_indices = y_val[y_val == 1].index
fraud_row = X_val.loc[[fraud_indices[0]]]
fraud_shap = explainer(fraud_row)

In [ ]:
shap.plots.waterfall(fraud_shap[0])

In [ ]:
import joblib
joblib.dump(model_xgb_final, "final_xgb_model.pkl")

In [ ]:
threshold = 0.2
joblib.dump(threshold, "fraud_threshold.pkl")

In [ ]:
print(type(loaded_model))
print(loaded_threshold)

In [ ]:
def predict_transaction(transaction):
    transaction_df = pd.DataFrame([transaction])
    fraud_probability = float(loaded_model.predict_proba(transaction_df)[:, 1][0])
    prediction = int(fraud_probability >= loaded_threshold)
    if prediction==1:
        label="FRAUD"
    else:
        label="LEGITIMATE"

    return {
        "fraud_probability": fraud_probability,
        "prediction": prediction,
        "label":label
    }

In [ ]:

test_dict = X_val.iloc[0].to_dict()
result = predict_transaction(test_dict)
print(result)

In [ ]:
import os
print(os.getcwd())

In [ ]:
loaded_model = joblib.load(
    "C:/Users/Ducks/Downloads/ducks python/fraud-detection-system/models/final_xgb_model.pkl"
)

In [ ]:
loaded_threshold = joblib.load(
    "C:/Users/Ducks/Downloads/ducks python/fraud-detection-system/models/fraud_threshold.pkl"
)
print(loaded_threshold)

In [ ]:
loaded_probability = float(loaded_model.predict_proba(test_transaction)[:, 1][0])
loaded_probability

In [ ]:

test_dict = X_val.iloc[0].to_dict()
result = predict_transaction(test_dict)
print(result)

In [ ]:
test_json = X_val.iloc[0].to_dict()
print(test_json)

In [ ]:
import json

print(json.dumps(test_json, indent=2))